# Clase 6 — De viajes observados a una API de predicción

En las dos clases anteriores definimos cómo una API recibe y valida una solicitud, pero la duración todavía provenía de una regla escrita a mano. Hoy completaremos la parte que falta: partiremos de viajes históricos, decidiremos qué información puede conocer el sistema antes de iniciar un viaje, entrenaremos una primera referencia y conectaremos ese modelo con FastAPI.

La pregunta que guía toda la clase es: **¿cómo podemos estimar la duración de un viaje nuevo y rastrear cada valor de la respuesta hasta una decisión sobre los datos?** El objetivo no es construir el mejor predictor posible, sino comprender el recorrido completo y poder reproducirlo.

## 1. El caso: estimar la duración de un viaje

Imagina una aplicación que conoce el origen, el destino, una distancia aproximada, la hora de salida y el número de pasajeros. Antes de comenzar el viaje quiere responder: **¿cuántos minutos podría durar?**

Para aprender esa relación necesitamos ejemplos de viajes terminados. La **Taxi and Limousine Commission (TLC)** es la agencia de la ciudad de Nueva York que regula servicios de taxi y publica registros mensuales de viajes. Cada fila describe un viaje observado e incluye, entre otros datos, hora de inicio, hora de término, distancia, pasajeros y zonas.

La duración real se conoce sólo cuando el viaje termina. Durante el entrenamiento será nuestra respuesta conocida o **target**. Las entradas disponibles antes de iniciar el viaje serán las **features**.

## 2. Los taxis y los archivos que usaremos

La **Taxi and Limousine Commission (TLC)** regula los taxis y otros servicios de transporte con licencia en Nueva York. En sus datos aparecen distintos tipos de servicio; para esta clase necesitamos distinguir dos:

- **Yellow Taxi:** es el taxi amarillo tradicional de Nueva York y opera con una licencia llamada *medallion*. Es el único tipo de taxi autorizado para recoger a una persona que lo detiene en la calle en cualquier punto de la ciudad.
- **Green Taxi:** también se conoce como *boro taxi* o *Street-Hail Livery*. Fue creado para ampliar ese servicio en los otros distritos y en el norte de Manhattan; no puede recoger pasajeros mediante *street hail* en el núcleo de Manhattan ni en los aeropuertos.

Trabajaremos con **Green Taxi** porque conserva las columnas necesarias para estimar la duración y sus archivos son mucho más pequeños. En marzo de 2026, el Parquet de Green pesa aproximadamente **1.1 MB**, frente a **67.9 MB** de Yellow. Eso permite descargar, explorar y entrenar durante una sesión. Esta elección es didáctica y computacional: Green no representa toda la movilidad de Nueva York. Fuentes: [Your Ride — TLC](https://www.nyc.gov/site/tlc/passengers/your-ride.page) y [Trip Records User Guide](https://www.nyc.gov/assets/tlc/downloads/pdf/trip_record_user_guide.pdf).

### ¿Por qué marzo y abril de 2026?

La TLC publica nuevos meses de manera periódica y con cierto retraso. Usar una ruta que significara “el archivo más reciente” cambiaría las filas, las métricas y las capturas durante el semestre. Por eso fijamos dos meses recientes, completos y consecutivos:

- **marzo de 2026:** entrenamiento;
- **abril de 2026:** validación con datos posteriores que el modelo no usó para aprender.

### Parquet frente a CSV

La [página oficial de Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) publica los archivos completos en **Parquet**.

| CSV | Parquet |
|---|---|
| texto separado por comas | formato binario y columnar |
| fácil de abrir en un editor | se abre con una biblioteca como pandas/pyarrow |
| los tipos deben interpretarse al leer | conserva tipos como fechas y números |
| suele ocupar más espacio | comprime mejor y permite leer columnas eficientemente |

## 3. Showcase: reconocer el caso en una muestra pequeña

Antes de trabajar con decenas de miles de viajes, reconozcamos la estructura del caso en un extracto de 24 filas. La muestra está en CSV para que sea pequeña y fácil de inspeccionar; no necesitamos copiarla porque sólo la leeremos desde el notebook.

Observa cuáles columnas provienen del archivo original y cuáles fueron preparadas para facilitar la conversación.

El notebook y la muestra viven en el repositorio público del curso. Este primer recorrido es una demostración: aquí no crearás ramas ni confirmarás cambios.

In [ ]:
import pandas as pd

muestra = pd.read_csv(
    "../labs/starters/clase-06-api-prediccion/muestra-green-taxi-2026-03.csv"
)
print(muestra.shape)
display(muestra.head())
muestra.dtypes

Ya reconocimos la estructura del caso. Ahora abre tu repositorio `pcd-entregas-2026`: allí construirás el análisis, el entrenamiento y la API que sí conservarás en tu historial. Mantén este notebook público abierto como guía. Los Parquet y los modelos generados permanecerán sólo en tu computadora; el código, las pruebas y tus explicaciones sí formarán parte de la entrega.

## 4. Preparar un solo espacio de trabajo

Desde la raíz de `pcd-entregas-2026` crea una rama para toda la clase:

```bash
git switch main
git pull
git switch -c feat/clase-06-api-prediccion
mkdir -p data/nyc-taxi actividades/clase-06-api-prediccion
```

Agrega `data/` al `.gitignore` de la raíz. Esta regla le indica a Git que no proponga los Parquet para un commit; los datos se pueden volver a descargar y no deben inflar el historial. Al final confirmaremos con `git status` y **Files changed** que los datos no aparezcan.

```gitignore
data/
```

Declara las bibliotecas que necesita la exploración:

```bash
uv add pandas pyarrow
```

Usaremos una sola rama y un solo PR. El EDA será el componente calificable, pero no fusionaremos nada a mitad de la clase: después continuaremos en esta misma carpeta con el modelo y la API.

## 5. Descargar marzo y abril con `curl`

```bash
curl --fail --location --progress-bar \
  --output data/nyc-taxi/green_tripdata_2026-03.parquet \
  https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-03.parquet

curl --fail --location --progress-bar \
  --output data/nyc-taxi/green_tripdata_2026-04.parquet \
  https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-04.parquet
```

- `--fail` hace visible una respuesta HTTP de error en lugar de tratar su contenido como dataset.
- Algunos servidores responden primero con una redirección HTTP, por ejemplo `301`, `302`, `307` o `308`, y colocan la siguiente dirección en el encabezado `Location`. `--location` indica a curl que siga esas redirecciones hasta llegar al archivo.
- `--output` determina el nombre y la ubicación local.
- `--progress-bar` muestra el avance sin llenar la terminal.

Comprueba que existan los dos archivos:

```bash
ls -lh data/nyc-taxi
```

### Alternativa desde la interfaz

Si `curl` no completa una descarga, abre la [página oficial de TLC](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), busca **Green Taxi Trip Records** y descarga marzo y abril de 2026 desde la interfaz. Mueve los archivos con Finder o el Explorador de archivos a `data/nyc-taxi/` y verifica que se llamen exactamente `green_tripdata_2026-03.parquet` y `green_tripdata_2026-04.parquet`.

## 6. Explorar marzo antes de elegir un modelo

Descargar los datos no basta. Antes de entrenar necesitamos decidir qué filas son confiables y qué variables estarían disponibles al comenzar un viaje. Queremos saber:

- qué representa una fila y qué columnas están disponibles;
- cómo construir la duración observada;
- cuántos viajes tienen duraciones y distancias plausibles;
- qué información falta;
- si una hipótesis como “el fin de semana cambia la duración” parece respaldada;
- qué variables podríamos conocer antes de comenzar un viaje.

Crea `actividades/clase-06-api-prediccion/eda.py`:

```python
from pathlib import Path

import pandas as pd

ARCHIVO_ACTUAL = Path(__file__).resolve()
RAIZ_REPO = ARCHIVO_ACTUAL.parents[2]
RUTA_DATOS = RAIZ_REPO / "data/nyc-taxi/green_tripdata_2026-03.parquet"
viajes = pd.read_parquet(RUTA_DATOS)

print(viajes.shape)
print(viajes.columns.tolist())
print(viajes.head())
print(viajes.dtypes)
```

`../../../` no corresponde a esta estructura: desde la carpeta de la actividad hacen falta dos niveles para llegar a la raíz, no tres. Además, una ruta como `../../data/...` depende de la carpeta desde la que ejecutes el comando. Aquí usamos `Path(__file__).resolve().parents[2]` para partir de la ubicación real de `eda.py` y encontrar `data/` aunque lances el script desde otra carpeta.

Completa la [Actividad de clase 6 — EDA de NYC Green Taxi](../docs/tareas/actividad-clase-06-eda.md). Realiza un primer commit cuando el EDA y su interpretación estén listos, pero **no cierres ni fusiones el PR todavía**.

## 7. Del EDA a una tabla para modelar

El EDA nos mostró qué columnas existen y qué registros requieren limpieza. Ahora el problema cambia: el modelo no puede recibir toda la tabla ni usar información que aparece sólo cuando el viaje terminó. Cada feature debe pasar tres preguntas:

1. ¿está disponible cuando se solicita la predicción?
2. ¿necesita una transformación para representar correctamente su significado?
3. ¿revela directa o indirectamente la duración que queremos predecir?

| Columna candidata | Decisión | Motivo |
|---|---|---|
| `trip_distance` | usar como `distancia_km` | TLC la reporta en millas; kilómetros son más familiares |
| `passenger_count` | usar como `pasajeros` | disponible, aunque cerca de 15.1 % de marzo está vacío |
| hora de pickup | usar como `hora_recoleccion` | disponible antes del viaje |
| `PULocationID`, `DOLocationID` | usar como zonas categóricas | origen y destino conocidos; aportan contexto espacial |
| fin de semana | conservar como hipótesis del EDA | permite comprobar si el patrón cambia entre días laborales y fin de semana |
| hora de dropoff | sólo construir el target | conocerla revelaría la respuesta |
| tarifa, propina, pago y peajes | descartar | se conocen durante o después del viaje |

## 8. Dos procesos que se conectan

El EDA termina en decisiones sobre filas y columnas. Ahora esas decisiones se reparten entre dos operaciones: primero construiremos y evaluaremos el modelo con datos históricos; después haremos que una API use el resultado para responder solicitudes nuevas.

```text
ENTRENAMIENTO — cuando construimos o actualizamos el modelo
Parquet histórico → preparación → ajuste → validación → artefacto entrenado

PREDICCIÓN — cada vez que llega una solicitud
JSON nuevo → validación → mismas features → modelo.predict() → JSON
```

La API no descarga los históricos ni vuelve a ajustar el modelo por cada solicitud. Recibe las cinco features del viaje nuevo y reutiliza el resultado del entrenamiento.

## 9. Preparar las mismas variables en marzo y abril

Crea `preparar_datos.py` en la carpeta de la actividad:

```python
from pathlib import Path

import pandas as pd

FEATURES_NUMERICAS = ["distancia_km", "pasajeros", "hora_recoleccion"]
FEATURES_CATEGORICAS = ["zona_origen", "zona_destino"]
FEATURES = FEATURES_NUMERICAS + FEATURES_CATEGORICAS
TARGET = "duracion_minutos"


def preparar_viajes(ruta: Path) -> pd.DataFrame:
    viajes = pd.read_parquet(ruta)
    viajes[TARGET] = (
        viajes["lpep_dropoff_datetime"] - viajes["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60
    viajes["distancia_km"] = viajes["trip_distance"] * 1.60934
    viajes["pasajeros"] = viajes["passenger_count"]
    viajes["hora_recoleccion"] = viajes["lpep_pickup_datetime"].dt.hour
    viajes["zona_origen"] = viajes["PULocationID"]
    viajes["zona_destino"] = viajes["DOLocationID"]

    validos = (
        viajes[TARGET].between(1, 60)
        & viajes["distancia_km"].between(0.1, 100)
        & viajes["pasajeros"].between(1, 6)
        & viajes["zona_origen"].gt(0)
        & viajes["zona_destino"].gt(0)
    )
    preparados = viajes.loc[validos, FEATURES + [TARGET]].dropna().copy()
    enteras = ["pasajeros", "hora_recoleccion", "zona_origen", "zona_destino"]
    preparados[enteras] = preparados[enteras].astype(int)
    return preparados
```

La función aplica las mismas decisiones a marzo y abril:

- resta la hora de inicio a la hora de término, convierte segundos a minutos con `/ 60` y obtiene el target `duracion_minutos`;
- multiplica `trip_distance` por **1.60934** porque TLC reporta millas y una milla equivale a 1.60934 kilómetros; así la feature coincide con la unidad que recibirá la API;
- renombra pasajeros para que el nombre del dataset y el contrato de entrada no queden mezclados;
- extrae la hora de recolección como un entero entre 0 y 23;
- renombra los identificadores de pickup y dropoff como zonas de origen y destino; aunque sean números, representan categorías geográficas;
- conserva duraciones, distancias, pasajeros y zonas dentro de rangos plausibles observados en el EDA;
- elimina valores faltantes de las columnas seleccionadas, crea una copia independiente y convierte las variables enteras a un tipo consistente.

Los límites de 1–60 minutos, 0.1–100 km y 1–6 pasajeros son decisiones explícitas para este ejercicio, no reglas universales sobre todos los viajes.

## 10. Entrenar, validar y conservar el resultado

Las variables numéricas pueden pasar al modelo con sus valores, pero los identificadores de zona necesitan tratamiento categórico: la zona 200 no es “el doble” de la zona 100. Construiremos un pipeline que prepare ambos tipos de columnas, ajuste una regresión lineal y mida el error sobre abril.

Al terminar el proceso de Python, el objeto aprendido desaparecería de la memoria. Como la API será otro proceso y no debe volver a entrenar, lo guardaremos como un **artefacto** local. Usaremos `pickle`, el formato binario de Python que puede serializar el pipeline y sus metadatos.

Ahora necesitamos scikit-learn, una carpeta para ese resultado y una regla que mantenga los artefactos fuera de Git:

```bash
uv add scikit-learn
mkdir -p artifacts/nyc-taxi
```

```gitignore
artifacts/
```

Crea `entrenar_modelo.py`:

```python
import pickle
from pathlib import Path

from preparar_datos import FEATURES, FEATURES_CATEGORICAS, TARGET, preparar_viajes
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RAIZ = Path(__file__).resolve().parents[2]
train = preparar_viajes(RAIZ / "data/nyc-taxi/green_tripdata_2026-03.parquet")
validacion = preparar_viajes(
    RAIZ / "data/nyc-taxi/green_tripdata_2026-04.parquet"
)
preprocesamiento = ColumnTransformer(
    [("zonas", OneHotEncoder(handle_unknown="ignore"), FEATURES_CATEGORICAS)],
    remainder="passthrough",
)
modelo = Pipeline(
    [("preprocesamiento", preprocesamiento), ("regresion", LinearRegression())]
).fit(train[FEATURES], train[TARGET])

predicciones = modelo.predict(validacion[FEATURES])
rmse = root_mean_squared_error(validacion[TARGET], predicciones)
artefacto = {
    "modelo": modelo,
    "features": FEATURES,
    "version": "green-taxi-2026-03-linear-zonas-1",
    "rmse_validacion": float(rmse),
}
ruta_modelo = RAIZ / "artifacts/nyc-taxi/modelo-duracion.pkl"
with ruta_modelo.open("wb") as archivo:
    pickle.dump(artefacto, archivo)

print(f"Entrenamiento: {len(train)} filas")
print(f"Validación: {len(validacion)} filas")
print(f"RMSE: {rmse:.2f} minutos")
```

Después del código, observa cómo se conectan las piezas:

- `OneHotEncoder` crea una columna indicadora por cada zona conocida; así evita imponer un orden numérico artificial.
- `ColumnTransformer` aplica esa codificación únicamente a origen y destino.
- `remainder="passthrough"` conserva distancia, pasajeros y hora sin aplicarles esa codificación.
- `handle_unknown="ignore"` evita que la validación o la API fallen si reciben una zona que marzo no observó; para esa zona, los indicadores conocidos quedan en cero.
- `Pipeline` mantiene juntos el preprocesamiento y la regresión, por lo que `fit` y `predict` siempre aplican la misma secuencia.
- el diccionario `artefacto` agrega el orden de las features, una versión y el RMSE al modelo entrenado antes de serializarlo.

Ejecuta desde la raíz:

```bash
uv run python actividades/clase-06-api-prediccion/entrenar_modelo.py
```

## 11. Ventajas y límites del artefacto `pickle`

El archivo `modelo-duracion.pkl` conserva lo necesario para que otro proceso reconstruya el objeto sin volver a ejecutar `fit`. Este mecanismo es práctico para el ejercicio, pero también introduce decisiones operativas.

| Ventajas | Limitaciones y riesgos |
|---|---|
| implementación local rápida | depende de Python y de versiones compatibles |
| conserva pipeline y regresión juntos | es opaco: Git no muestra un diff útil |
| carga rápida y sin servicio externo | puede ocupar espacio y duplicarse en el historial |
| separa entrenamiento de predicción | cargar un pickle no confiable puede ejecutar código |

Por eso el pickle se genera localmente y `artifacts/` no entra a Git. Sí confirmamos el script y las dependencias que permiten reconstruirlo. Nunca cargamos un pickle recibido de una fuente desconocida.

## 12. Construir la parte de predicción con FastAPI

Ahora comenzamos el segundo proceso: **predicción**. Crea `main.py`. La aplicación carga el pickle una vez al iniciar, fuera del endpoint; después cada solicitud reutiliza el mismo objeto.

```python
import pickle
from pathlib import Path

import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

RAIZ = Path(__file__).resolve().parents[2]
RUTA_MODELO = RAIZ / "artifacts/nyc-taxi/modelo-duracion.pkl"
with RUTA_MODELO.open("rb") as archivo:
    artefacto = pickle.load(archivo)


class SolicitudPrediccion(BaseModel):
    distancia_km: float = Field(gt=0, le=100)
    pasajeros: int = Field(ge=1, le=6)
    hora_recoleccion: int = Field(ge=0, le=23)
    zona_origen: int = Field(ge=1, le=265)
    zona_destino: int = Field(ge=1, le=265)


class RespuestaPrediccion(BaseModel):
    duracion_estimada_minutos: float
    version_modelo: str


app = FastAPI(title="API de duración de viajes Green Taxi")


@app.post("/predicciones", response_model=RespuestaPrediccion)
def crear_prediccion(solicitud: SolicitudPrediccion) -> RespuestaPrediccion:
    entrada = pd.DataFrame([solicitud.model_dump()])[artefacto["features"]]
    duracion = float(artefacto["modelo"].predict(entrada)[0])
    return RespuestaPrediccion(
        duracion_estimada_minutos=round(duracion, 1),
        version_modelo=artefacto["version"],
    )
```

Pydantic valida la forma del viaje nuevo. El pipeline transforma las zonas y calcula la predicción. La lista guardada en `artefacto["features"]` conserva exactamente el orden usado al entrenar.

## 13. Probar el recorrido completo

Desde la raíz del repositorio:

```bash
cd actividades/clase-06-api-prediccion
uv run fastapi dev
```

### Opción A: Postman

1. Crea una solicitud con método `POST`.
2. Usa la URL `http://127.0.0.1:8000/predicciones`.
3. En **Body**, selecciona **raw** y después **JSON**.
4. Envía este cuerpo:

```json
{
  "distancia_km": 4.2,
  "pasajeros": 2,
  "hora_recoleccion": 18,
  "zona_origen": 75,
  "zona_destino": 42
}
```

Postman añadirá `Content-Type: application/json` al elegir JSON. Presiona **Send** y verifica la respuesta.

### Opción B: `curl`

En otra terminal:

```bash
curl -i -X POST http://127.0.0.1:8000/predicciones \
  -H 'Content-Type: application/json' \
  -d '{"distancia_km": 4.2, "pasajeros": 2, "hora_recoleccion": 18, "zona_origen": 75, "zona_destino": 42}'
```

Después cambia `hora_recoleccion` a `24` y luego omite `zona_destino`. Ambas solicitudes deben recibir `422` antes de ejecutar el predictor. Repite la solicitud válida en `http://127.0.0.1:8000/docs`.

Un `200` demuestra que el recorrido técnico funciona. No demuestra por sí solo que el modelo sea exacto, justo, estable o adecuado para producción.

## 14. Cerrar el trabajo de la clase

Todo permanece en la misma rama. Si todavía no hiciste el checkpoint del EDA, confírmalo primero; después confirma el recorrido guiado:

```bash
git add .gitignore pyproject.toml uv.lock \
  actividades/clase-06-api-prediccion/eda.py \
  actividades/clase-06-api-prediccion/README.md
git commit -m "feat(clase-06): explora viajes green taxi"

git add .gitignore pyproject.toml uv.lock \
  actividades/clase-06-api-prediccion/preparar_datos.py \
  actividades/clase-06-api-prediccion/entrenar_modelo.py \
  actividades/clase-06-api-prediccion/main.py
git commit -m "feat(clase-06): entrena y expone modelo de duracion"

git status --short
git push -u origin feat/clase-06-api-prediccion
```

Ajusta cada `git add` a los archivos realmente completados, incluida la evidencia que hayas agregado al README. Revisa en `git status` y después en **Files changed** que no aparezcan `.parquet` ni `.pkl`. Abre un solo PR hacia `main`, revisa la plantilla, fusiónalo y entrega en Canvas la URL con estado **Merged**.

La rúbrica de la actividad califica el EDA y su interpretación. El código posterior permanece en el mismo PR porque documenta el recorrido construido en clase, pero no agrega criterios ocultos a esa rúbrica.

## Para la siguiente clase

Queda asignada la [Tarea 3 — comparación de dos modelos servidos por una API](../docs/tareas/tarea-03-api-prediccion.md), con entrega el **lunes 7 de septiembre de 2026 a las 19:55**. Comienza en una rama nueva sólo después de fusionar el PR de hoy. Reproducirás el modelo lineal, entrenarás un bosque aleatorio, expondrás ambos con endpoints distintos y compararás sus métricas. Los Parquet y los archivos `.pkl` permanecen locales y no se suben a Git.

La Clase 7 comenzará con el **Quiz 2** el **lunes 7 de septiembre de 2026 a las 20:10**. Tendrás **15 minutos**. Repasa Clases 4–6: HTTP y FastAPI, contratos Pydantic, preparación y selección de features, diferencia entre entrenamiento y predicción, carga de pickle y respuestas `200`/`422`.

Antes de cerrar confirma que puedes explicar por qué:

- marzo enseña y abril valida;
- una zona necesita codificación categórica;
- el tiempo de llegada no es una feature válida;
- entrenamiento y predicción son procesos distintos;
- el pickle no entra a Git, pero el código que lo reproduce sí.